# 📊 Morning Call Digest → Email Diário

Coleta transcrições dos principais morning calls, resume em 3-4 parágrafos com Gemini e envia por email automaticamente.

```
Último dia útil → transcrições (BTG, Genial, Investing, XP) → Gemini resume → Email
```

## 1. Instalação

In [1]:
!pip install youtube-transcript-api google-genai -q
!pip install --upgrade yt-dlp -q

import subprocess
v = subprocess.run(['yt-dlp', '--version'], capture_output=True, text=True)
print(f'✅ yt-dlp versão: {v.stdout.strip()}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.3 MB/s eta 0:00:00
✅ yt-dlp versão: 2026.03.17


## 2. Configuração

In [2]:
from google import genai
from google.genai import types
from youtube_transcript_api import YouTubeTranscriptApi
import re, json, subprocess, shutil, sys, smtplib
from datetime import datetime, date, timedelta
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# ================================================================
# 👉 Configurações — preencha aqui
# ================================================================

# from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GEMINI_API_KEY = 'sua-chave-gemini'

# Email — use senha de app do Gmail
# Como gerar: myaccount.google.com → Segurança → Senhas de app
EMAIL_REMETENTE    = 'seu-email@gmail.com'
EMAIL_SENHA_APP    = 'sua-senha-de-app'
DESTINATARIOS      = ['destino@gmail.com']  # adicione quantos quiser

MODELO_PRINCIPAL = 'gemini-3.1-flash-lite'

client    = genai.Client(api_key=GEMINI_API_KEY)
YTDLP_CMD = shutil.which('yt-dlp') or [sys.executable, '-m', 'yt_dlp']
if isinstance(YTDLP_CMD, str):
    YTDLP_CMD = [YTDLP_CMD]

try:
    r = client.models.generate_content(model=MODELO_PRINCIPAL, contents='responda só: ok')
    print(f'✅ Gemini conectado | modelo: {MODELO_PRINCIPAL}')
except Exception as e:
    print(f'❌ Erro Gemini: {e}')

print(f'🔧 yt-dlp: {YTDLP_CMD}')

❌ Erro Gemini: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}
🔧 yt-dlp: ['/usr/local/bin/yt-dlp']


## 3. Canais, busca e extração de transcrição

In [3]:
CANAIS_MORNING_CALL = {
    'btg':       {'url': 'https://www.youtube.com/@BTGPactual/streams',         'keywords': ['morning call', 'btg pactual']},
    'genial':    {'url': 'https://www.youtube.com/@genialinvestimentos/streams', 'keywords': ['resumo da manhã', 'morning call genial']},
    'investing': {'url': 'https://www.youtube.com/@investingcombr/streams',     'keywords': ['morning call']},
    'xp':        {'url': 'https://www.youtube.com/@XP_Oficial/streams',         'keywords': ['morning call']},
}

PROMPT_SELECAO = """
Você é um assistente especializado em conteúdo financeiro do YouTube brasileiro.

Sua tarefa é identificar qual vídeo da lista abaixo é o MORNING CALL do dia {data}.

O morning call é um programa jornalístico de mercado transmitido toda manhã antes ou
durante a abertura do pregão. Cada canal pode usar nomes diferentes:
- "Morning Call", "Resumo da Manhã", "Abertura de Mercado", "Panorama do Dia"
- Título com a data do dia + assuntos do mercado (ex: "22/05 Petróleo cai 5%...")

NÃO é morning call:
- Day Trade ao vivo (operações em tempo real, geralmente +2h)
- Fechamento de mercado
- Análise de ações específicas sem contexto macro do dia
- Tutoriais ou conteúdo educacional
- Entrevistas temáticas sem foco no panorama do dia

IMPORTANTE:
- O vídeo DEVE ser do dia {data} (formato YYYYMMDD = {data_yyyymmdd})
- Se nenhum vídeo do dia for morning call, retorne null no id
- Escolha apenas UM vídeo

Retorne SOMENTE JSON puro:
{{
  "escolhido": {{
    "id": "video_id ou null",
    "titulo": "título ou null",
    "motivo": "1 frase explicando a escolha"
  }},
  "descartados": [
    {{"id": "video_id", "motivo": "motivo do descarte"}}
  ]
}}

VÍDEOS DISPONÍVEIS:
{videos_json}
"""


def ultimo_dia_util() -> date:
    """Retorna o dia útil mais recente com morning call disponível.
    Se for dia útil e já passou das 10h, retorna hoje.
    Caso contrário, retorna o último dia útil anterior.
    """
    agora = datetime.now()
    hoje  = agora.date()
    if hoje.weekday() < 5 and agora.hour >= 10:
        return hoje
    d = hoje - timedelta(days=1)
    while d.weekday() >= 5:
        d -= timedelta(days=1)
    return d


def buscar_video_canal(nome_canal: str, config: dict,
                        dia_alvo: date, max_busca: int = 20) -> tuple:
    """
    Busca vídeos do canal, filtra pelo dia exato e usa IA para identificar o morning call.
    NUNCA retorna vídeo de data diferente do dia_alvo.
    Retorna (video, None) ou (None, motivo_falha).
    """
    dia_str = dia_alvo.strftime('%Y%m%d')

    cmd = YTDLP_CMD + [
        '--flat-playlist', '--playlist-end', str(max_busca),
        '--print', '%(id)s\t%(title)s\t%(duration)s',
        '--no-warnings', config['url']
    ]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=60)

    if not r.stdout.strip():
        return None, 'canal inacessível ou sem vídeos'

    videos = []
    for linha in r.stdout.strip().split('\n'):
        partes = linha.split('\t')
        if len(partes) < 2:
            continue
        try:
            duracao = int(float(partes[2])) if len(partes) > 2 else 0
        except Exception:
            duracao = 0
        videos.append({
            'id':          partes[0].strip(),
            'titulo':      partes[1].strip(),
            'data':        '',
            'duracao_seg': duracao,
            'url':         f'https://youtube.com/watch?v={partes[0].strip()}'
        })

    # Busca datas — release_date tem prioridade para canais de live
    for v in videos:
        try:
            r2 = subprocess.run(
                YTDLP_CMD + ['--no-warnings', '--skip-download',
                             '--print', '%(release_date)s\t%(upload_date)s', v['url']],
                capture_output=True, text=True, timeout=30
            )
            partes_data = r2.stdout.strip().split('\t')
            release = partes_data[0].strip() if len(partes_data) > 0 else ''
            upload  = partes_data[1].strip() if len(partes_data) > 1 else ''
            if release and release not in ('NA', 'None', ''):
                v['data'] = release
            elif upload and upload not in ('NA', 'None', ''):
                v['data'] = upload
            else:
                v['data'] = ''
        except Exception:
            v['data'] = ''

    # Filtra apenas vídeos do dia exato — sem fallback de data
    videos_do_dia = [v for v in videos if v['data'] == dia_str]

    if not videos_do_dia:
        ultimo      = max((v for v in videos if v['data']), key=lambda x: x['data'], default=None)
        ultima_data = ultimo['data'] if ultimo else 'desconhecida'
        return None, f'nenhum vídeo publicado em {dia_alvo.strftime("%d/%m/%Y")} (último disponível: {ultima_data})'

    print(f'   📋 {len(videos_do_dia)} vídeo(s) do dia — IA selecionando morning call...')

    # IA seleciona o morning call entre os vídeos do dia
    videos_slim = [{
        'id':      v['id'],
        'titulo':  v['titulo'],
        'duracao': f'{v["duracao_seg"]//60}min',
        'data':    v['data']
    } for v in videos_do_dia]

    prompt = PROMPT_SELECAO.format(
        data=dia_alvo.strftime('%d/%m/%Y'),
        data_yyyymmdd=dia_str,
        videos_json=json.dumps(videos_slim, ensure_ascii=False, indent=2)
    )

    resposta = client.models.generate_content(
        model=MODELO_PRINCIPAL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=512,
            response_mime_type='application/json'
        )
    )

    texto     = re.sub(r'^```json\s*|\s*```$', '', resposta.text.strip(), flags=re.MULTILINE).strip()
    resultado = json.loads(texto)

    escolhido_id = resultado.get('escolhido', {}).get('id')
    motivo_ia    = resultado.get('escolhido', {}).get('motivo', '')

    if not escolhido_id or escolhido_id in ('null', None):
        return None, 'IA não identificou morning call entre os vídeos do dia'

    video = next((v for v in videos_do_dia if v['id'] == escolhido_id), None)
    if not video:
        return None, f'IA escolheu id {escolhido_id} mas não encontrado na lista'

    if video['duracao_seg'] < 300:
        return None, f'vídeo selecionado muito curto ({video["duracao_seg"]}s) — provável live em andamento'

    print(f'   ✅ [{nome_canal}] {video["titulo"][:55]} ({video["duracao_seg"]//60}min)')
    print(f'      Motivo IA: {motivo_ia}')
    return video, None


def obter_transcricao(url_ou_id: str, idiomas: list = ['pt', 'pt-BR', 'en']) -> dict:
    video_id = (url_ou_id if len(url_ou_id) == 11
                else re.search(r'(?:v=|youtu\.be/)([\w-]{11})', url_ou_id).group(1))
    print(f'   🎬 Transcrição: {video_id}')

    ytt   = YouTubeTranscriptApi()
    lista = ytt.list(video_id)

    transcript = None
    for metodo in [
        lambda: lista.find_manually_created_transcript(idiomas),
        lambda: lista.find_generated_transcript(idiomas),
        lambda: next(iter(lista))
    ]:
        try:
            transcript = metodo()
            break
        except Exception:
            continue

    segmentos = transcript.fetch()
    texto     = ' '.join(s.text for s in segmentos)
    duracao   = int(segmentos[-1].start + (segmentos[-1].duration or 0))

    return {'video_id': video_id, 'texto_completo': texto, 'duracao_segundos': duracao}


print('✅ Canais e funções de extração carregadas')

✅ Canais e funções de extração carregadas


## 4. Resumo com Gemini

In [4]:
PROMPT_RESUMO = """
Você é um analista de mercado. Recebeu as transcrições de {n_canais} morning calls
de {data} dos canais: {canais_presentes}.

{aviso_ausentes}

Escreva um resumo executivo em exatamente 3 a 4 parágrafos para ser enviado por email
a um investidor que não teve tempo de assistir os vídeos.

REGRAS:
- Use APENAS informações explicitamente presentes nas transcrições. NUNCA invente.
- Sempre atribua informações ao canal: "Segundo o BTG...", "A Genial destaca..."
- Consenso: "Há consenso entre [canais] de que..."
- Divergência: "Enquanto [canal1] vê X, [canal2] aponta Y."
- Se canais estiverem ausentes, mencione brevemente no primeiro parágrafo.
- Priorize: cenário macro, movimentos de mercado, ativos em destaque, riscos.
- Sem bullet points, sem títulos, apenas texto corrido.
- Não comece com "Bom dia" ou introduções genéricas.

TRANSCRIÇÕES:
---
{transcricoes}
---

Escreva o resumo agora:
"""


def gerar_resumo(transcricoes_por_canal: dict, canais_ausentes: dict,
                  data_ref: str, modelo: str = MODELO_PRINCIPAL) -> str:
    bloco = ''
    for canal, texto in transcricoes_por_canal.items():
        MAX_CHARS  = 40_000
        texto_trim = texto[:MAX_CHARS] + '...[truncado]' if len(texto) > MAX_CHARS else texto
        bloco += f'\n[{canal.upper()}]\n{texto_trim}\n'

    if canais_ausentes:
        linhas = '\n'.join(f'- {c.upper()}: {m}' for c, m in canais_ausentes.items())
        aviso  = (f'ATENÇÃO: Os seguintes canais NÃO estão disponíveis hoje:\n{linhas}\n'
                  f'Mencione brevemente no primeiro parágrafo.')
    else:
        aviso = ''

    prompt = PROMPT_RESUMO.format(
        n_canais=len(transcricoes_por_canal),
        data=data_ref,
        canais_presentes=', '.join(c.upper() for c in transcricoes_por_canal.keys()),
        aviso_ausentes=aviso,
        transcricoes=bloco
    )

    print(f'🤖 Gerando resumo com Gemini ({modelo})...')

    resposta = client.models.generate_content(
        model=modelo,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            max_output_tokens=1024,
        )
    )

    resumo = resposta.text.strip()
    print(f'✅ Resumo gerado | {len(resumo):,} chars')
    return resumo


print('✅ Função de resumo carregada')

✅ Função de resumo carregada


## 5. Email

In [5]:
def montar_html(resumo: str, data_ref: str, canais_usados: list,
                 canais_ausentes: dict, videos_info: list) -> str:
    paragrafos = ''.join(
        f'<p style="margin:0 0 14px 0;">{p.strip()}</p>'
        for p in resumo.split('\n') if p.strip()
    )
    fontes = ''.join(
        f'<li><a href="{v["url"]}" style="color:#1a73e8;">[{v["canal"].upper()}] {v["titulo"][:70]}</a></li>'
        for v in videos_info
    )
    bloco_ausentes = ''
    if canais_ausentes:
        itens = ''.join(f'<li><b>{c.upper()}</b>: {m}</li>' for c, m in canais_ausentes.items())
        bloco_ausentes = f"""
        <div style="background:#fff8e1;border-left:4px solid #f9a825;
                    padding:12px 16px;margin-bottom:16px;border-radius:4px;">
          <p style="margin:0 0 6px 0;font-size:13px;font-weight:bold;color:#f57f17;">
            ⚠️ Canais ausentes hoje</p>
          <ul style="margin:0;padding-left:16px;font-size:12px;color:#555;">{itens}</ul>
        </div>"""

    return f"""
    <html><body style="font-family:Arial,sans-serif;max-width:680px;margin:auto;color:#222;">
      <div style="background:#1a1a2e;padding:20px 28px;border-radius:8px 8px 0 0;">
        <h2 style="color:#fff;margin:0;font-size:18px;">📊 Morning Call Digest</h2>
        <p style="color:#aaa;margin:4px 0 0 0;font-size:13px;">
          {data_ref} &nbsp;·&nbsp; {', '.join(c.upper() for c in canais_usados)}</p>
      </div>
      <div style="background:#f9f9f9;padding:24px 28px;border:1px solid #e0e0e0;border-top:none;">
        {bloco_ausentes}{paragrafos}
      </div>
      <div style="padding:16px 28px;border:1px solid #e0e0e0;border-top:none;background:#fff;">
        <p style="font-size:12px;color:#888;margin:0 0 8px 0;">Fontes:</p>
        <ul style="font-size:12px;color:#555;margin:0;padding-left:18px;">{fontes}</ul>
      </div>
      <p style="font-size:11px;color:#bbb;text-align:center;margin-top:12px;">
        Gerado automaticamente · {datetime.now().strftime('%d/%m/%Y %H:%M')}</p>
    </body></html>"""


def enviar_emails(resumo: str, data_ref: str, canais_usados: list,
                   canais_ausentes: dict, videos_info: list) -> bool:
    n_ausentes = len(canais_ausentes)
    sufixo     = f' — ⚠️ {n_ausentes} canal(is) ausente(s)' if n_ausentes else ''
    assunto    = f'📊 Morning Call Digest — {data_ref}{sufixo}'
    html       = montar_html(resumo, data_ref, canais_usados, canais_ausentes, videos_info)

    msg = MIMEMultipart('alternative')
    msg['Subject'] = assunto
    msg['From']    = EMAIL_REMETENTE
    msg['To']      = ', '.join(DESTINATARIOS)
    msg.attach(MIMEText(resumo, 'plain', 'utf-8'))
    msg.attach(MIMEText(html,   'html',  'utf-8'))

    try:
        print(f'📧 Enviando email para: {", ".join(DESTINATARIOS)}...')
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(EMAIL_REMETENTE, EMAIL_SENHA_APP)
            smtp.sendmail(EMAIL_REMETENTE, DESTINATARIOS, msg.as_string())
        print(f'✅ Email enviado!')
        return True
    except Exception as e:
        print(f'❌ Erro ao enviar email: {e}')
        return False


def enviar_email_sem_digest(data_ref: str, canais_ausentes: dict) -> bool:
    itens = ''.join(f'<li><b>{c.upper()}</b>: {m}</li>' for c, m in canais_ausentes.items())
    html  = f"""
    <html><body style="font-family:Arial,sans-serif;max-width:680px;margin:auto;color:#222;">
      <div style="background:#1a1a2e;padding:20px 28px;border-radius:8px 8px 0 0;">
        <h2 style="color:#fff;margin:0;font-size:18px;">📊 Morning Call Digest</h2>
        <p style="color:#aaa;margin:4px 0 0 0;font-size:13px;">{data_ref}</p>
      </div>
      <div style="background:#f9f9f9;padding:24px 28px;border:1px solid #e0e0e0;border-top:none;">
        <div style="background:#ffebee;border-left:4px solid #c62828;
                    padding:12px 16px;border-radius:4px;">
          <p style="margin:0 0 8px 0;font-weight:bold;color:#b71c1c;">
            ❌ Nenhum morning call disponível para {data_ref}</p>
          <ul style="font-size:13px;color:#555;margin:0;padding-left:16px;">{itens}</ul>
        </div>
      </div>
      <p style="font-size:11px;color:#bbb;text-align:center;margin-top:12px;">
        Gerado automaticamente · {datetime.now().strftime('%d/%m/%Y %H:%M')}</p>
    </body></html>"""

    msg = MIMEMultipart('alternative')
    msg['Subject'] = f'📊 Morning Call Digest — {data_ref} — ❌ Sem conteúdo disponível'
    msg['From']    = EMAIL_REMETENTE
    msg['To']      = ', '.join(DESTINATARIOS)
    msg.attach(MIMEText(f'Nenhum morning call disponível para {data_ref}.', 'plain', 'utf-8'))
    msg.attach(MIMEText(html, 'html', 'utf-8'))

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(EMAIL_REMETENTE, EMAIL_SENHA_APP)
            smtp.sendmail(EMAIL_REMETENTE, DESTINATARIOS, msg.as_string())
        print('✅ Email de aviso enviado!')
        return True
    except Exception as e:
        print(f'❌ Erro ao enviar email de aviso: {e}')
        return False


print('✅ Funções de email carregadas')

✅ Funções de email carregadas


## 6. Pipeline completo

In [6]:
def pipeline_digest(canais: dict = CANAIS_MORNING_CALL, enviar: bool = True) -> dict:
    """
    Pipeline completo:
    1. Detecta automaticamente o último dia útil com morning call disponível
    2. Busca o morning call de cada canal para esse dia
    3. Extrai transcrições
    4. Gemini resume em 3-4 parágrafos
    5. Envia por email para todos os destinatários em DESTINATARIOS

    Tratamento de erros:
    - Canal ausente: mencionado no resumo e no email, mas não impede o envio
    - 0 canais disponíveis: envia email de aviso sem resumo
    - Vídeo de data errada: descartado automaticamente, nunca entra no resumo

    enviar=False para testar sem mandar o email.
    """
    dia      = ultimo_dia_util()
    data_ref = dia.strftime('%d/%m/%Y')
    print(f'🗓️  Digest de: {data_ref}\n')

    transcricoes_por_canal = {}
    canais_ausentes        = {}
    videos_info            = []

    # 1. Coleta transcrições
    for nome, config in canais.items():
        print(f'📡 [{nome}]')
        try:
            video, motivo_falha = buscar_video_canal(nome, config, dia_alvo=dia)

            if not video:
                canais_ausentes[nome] = motivo_falha
                print(f'   ⚠️  Ausente: {motivo_falha}\n')
                continue

            transcricao = obter_transcricao(video['url'])
            transcricoes_por_canal[nome] = transcricao['texto_completo']
            videos_info.append({
                'canal':  nome,
                'titulo': video['titulo'],
                'url':    video['url']
            })
            print(f'   ✅ OK | {transcricao["duracao_segundos"]//60}min | '
                  f'{len(transcricao["texto_completo"]):,} chars\n')

        except Exception as e:
            motivo = str(e)[:120]
            canais_ausentes[nome] = f'erro técnico: {motivo}'
            print(f'   ❌ Erro: {motivo}\n')
            continue

    # 2. Avalia resultado
    coletados = len(transcricoes_por_canal)
    total     = len(canais)
    print(f'📊 {coletados}/{total} canais coletados')
    if canais_ausentes:
        for canal, motivo in canais_ausentes.items():
            print(f'   ⚠️  [{canal}] {motivo}')
    print()

    # 0 canais → email de aviso
    if coletados == 0:
        print('❌ Nenhum canal disponível — enviando email de aviso')
        if enviar:
            enviar_email_sem_digest(data_ref, canais_ausentes)
        else:
            print('ℹ️  enviar=False — email de aviso não enviado')
        return {'data': data_ref, 'canais': [], 'ausentes': canais_ausentes}

    # 3. Gera resumo
    resumo = gerar_resumo(transcricoes_por_canal, canais_ausentes, data_ref)

    print(f'\n{"-"*60}')
    print('📄 PRÉVIA DO RESUMO:')
    print(f'{"-"*60}')
    print(resumo)
    print(f'{"-"*60}\n')

    # 4. Envia email
    if enviar:
        enviar_emails(resumo, data_ref,
                      list(transcricoes_por_canal.keys()),
                      canais_ausentes,
                      videos_info)
    else:
        print('ℹ️  enviar=False — email não enviado (modo teste)')

    return {
        'data':        data_ref,
        'canais':      list(transcricoes_por_canal.keys()),
        'ausentes':    canais_ausentes,
        'resumo':      resumo,
        'videos_info': videos_info
    }


print('✅ Pipeline completo carregado')

✅ Pipeline completo carregado


## 7. Execução

> **Primeiro teste:** deixe `enviar=False` para ver o resumo sem mandar o email.  
> Quando estiver satisfeito, troque para `enviar=True`.

In [7]:
resultado = pipeline_digest(enviar=False)

🗓️  Digest de: 25/05/2026

📡 [btg]
   ⚠️  Ausente: nenhum vídeo publicado em 25/05/2026 (último disponível: desconhecida)

📡 [genial]
   ⚠️  Ausente: nenhum vídeo publicado em 25/05/2026 (último disponível: desconhecida)

📡 [investing]
   ⚠️  Ausente: nenhum vídeo publicado em 25/05/2026 (último disponível: desconhecida)

📡 [xp]
   ⚠️  Ausente: nenhum vídeo publicado em 25/05/2026 (último disponível: desconhecida)

📊 0/4 canais coletados
   ⚠️  [btg] nenhum vídeo publicado em 25/05/2026 (último disponível: desconhecida)
   ⚠️  [genial] nenhum vídeo publicado em 25/05/2026 (último disponível: desconhecida)
   ⚠️  [investing] nenhum vídeo publicado em 25/05/2026 (último disponível: desconhecida)
   ⚠️  [xp] nenhum vídeo publicado em 25/05/2026 (último disponível: desconhecida)

❌ Nenhum canal disponível — enviando email de aviso
ℹ️  enviar=False — email de aviso não enviado
